# E-Commerce Data Analysis — SQL Analysis
### Tools: Python, SQLite, Pandas
---
## Overview
This notebook loads the cleaned dataset into a SQLite database and runs
SQL queries to answer key business questions. SQL is used as it reflects
real world analyst workflows where data lives in databases not CSV files.

import pandas as pd
import numpy as np
import sqlite3
import matplotlib.pyplot as plt
import seaborn as sns

df = pd.read_csv(r'C:\Users\Guganathan\Desktop\ecommerce-data-analysis\data\data_cleaned.csv')
df['InvoiceDate'] = pd.to_datetime(df['InvoiceDate'])

print("Data loaded! Shape:", df.shape)

## Setup — Create SQLite Database
We load the cleaned CSV into a SQLite database table called 'orders'.
This simulates a real company database environment.

In [2]:
# Create SQLite database and load data
conn = sqlite3.connect(r'C:\Users\Guganathan\Desktop\ecommerce-data-analysis\data\ecommerce.db')
df.to_sql('orders', conn, if_exists='replace', index=False)
print("Database created and data loaded successfully!")
print("Total rows in database:", pd.read_sql("SELECT COUNT(*) as total FROM orders", conn).iloc[0,0])

Database created and data loaded successfully!
Total rows in database: 391295


## Query 1 — Top 10 Customers by Revenue
Who are the highest value customers and how many orders have they placed?

In [3]:
# Query 1 - Top 10 customers by revenue
query1 = """
SELECT CustomerID, 
       ROUND(SUM(TotalPrice), 2) as TotalRevenue,
       COUNT(DISTINCT InvoiceNo) as TotalOrders
FROM orders
GROUP BY CustomerID
ORDER BY TotalRevenue DESC
LIMIT 10
"""
result1 = pd.read_sql(query1, conn)
print("Top 10 Customers by Revenue:")
print(result1)

Top 10 Customers by Revenue:
   CustomerID  TotalRevenue  TotalOrders
0       14646     279138.02           72
1       18102     259657.30           60
2       17450     194390.79           46
3       14911     140336.83          199
4       12415     124564.53           20
5       14156     117210.08           55
6       17511      91062.38           31
7       16029      72708.09           62
8       16684      66653.56           28
9       13694      65039.62           50


## Query 2 — Revenue by Country
Which countries generate the most revenue, customers, and orders?

In [4]:
# Query 2 - Revenue by Country
query2 = """
SELECT Country,
       ROUND(SUM(TotalPrice), 2) as TotalRevenue,
       COUNT(DISTINCT CustomerID) as TotalCustomers,
       COUNT(DISTINCT InvoiceNo) as TotalOrders
FROM orders
GROUP BY Country
ORDER BY TotalRevenue DESC
LIMIT 10
"""
result2 = pd.read_sql(query2, conn)
print("Revenue by Country:")
print(result2)

Revenue by Country:
          Country  TotalRevenue  TotalCustomers  TotalOrders
0  United Kingdom    6959872.12            3916        16587
1     Netherlands     283889.34               9           93
2            EIRE     261888.12               3          258
3         Germany     205381.15              94          443
4          France     183987.94              87          380
5       Australia     138103.81               9           56
6           Spain      55706.56              30           88
7     Switzerland      52441.95              21           47
8           Japan      37416.37               8           19
9         Belgium      36927.34              25           98


## Query 3 — Monthly Revenue
How does revenue, order count, and active customers change month by month?

In [5]:
# Query 3 - Monthly Revenue
query3 = """
SELECT Year,
       Month,
       ROUND(SUM(TotalPrice), 2) as MonthlyRevenue,
       COUNT(DISTINCT InvoiceNo) as TotalOrders,
       COUNT(DISTINCT CustomerID) as ActiveCustomers
FROM orders
GROUP BY Year, Month
ORDER BY Year, Month
"""
result3 = pd.read_sql(query3, conn)
print("Monthly Revenue:")
print(result3)

Monthly Revenue:
    Year  Month  MonthlyRevenue  TotalOrders  ActiveCustomers
0   2010     12       565764.56         1396              885
1   2011      1       485714.31          983              738
2   2011      2       442493.59          992              757
3   2011      3       583843.85         1312              973
4   2011      4       454855.88         1140              853
5   2011      5       659693.49         1545             1055
6   2011      6       614794.92         1389              990
7   2011      7       592103.79         1321              946
8   2011      8       635864.38         1268              933
9   2011      9       939682.63         1742             1261
10  2011     10      1003056.56         1905             1361
11  2011     11      1137664.00         2644             1661
12  2011     12       343923.48          776              614


## Query 4 — Top 10 Products
Which products generate the most revenue and what is their average price?

In [6]:
# Query 4 - Top 10 products by revenue
query4 = """
SELECT Description,
       ROUND(SUM(TotalPrice), 2) as TotalRevenue,
       SUM(Quantity) as TotalQuantity,
       ROUND(AVG(UnitPrice), 2) as AvgPrice
FROM orders
GROUP BY Description
ORDER BY TotalRevenue DESC
LIMIT 10
"""
result4 = pd.read_sql(query4, conn)
print("Top 10 Products:")
print(result4)

Top 10 Products:
                          Description  TotalRevenue  TotalQuantity  AvgPrice
0            REGENCY CAKESTAND 3 TIER     142264.75          12374     12.48
1  WHITE HANGING HEART T-LIGHT HOLDER     100392.10          36706      2.89
2             JUMBO BAG RED RETROSPOT      85040.54          46078      2.02
3                       PARTY BUNTING      68785.23          15279      4.88
4       ASSORTED COLOUR BIRD ORNAMENT      56413.03          35263      1.68
5                  RABBIT NIGHT LIGHT      51251.24          27153      2.01
6                       CHILLI LIGHTS      46265.11           9646      5.43
7     PAPER CHAIN KIT 50'S CHRISTMAS       42584.13          15591      2.94
8            BLACK RECORD COVER FRAME      39045.80          11401      3.66
9             JUMBO BAG PINK POLKADOT      37254.36          20148      2.02


## Query 5 — Customer Segments
How are customers distributed across order frequency segments?

In [7]:
# Query 5 - Customer order frequency segments
query5 = """
SELECT 
    CASE 
        WHEN TotalOrders = 1 THEN '1 order'
        WHEN TotalOrders BETWEEN 2 AND 5 THEN '2-5 orders'
        WHEN TotalOrders BETWEEN 6 AND 10 THEN '6-10 orders'
        WHEN TotalOrders > 10 THEN '10+ orders'
    END as CustomerSegment,
    COUNT(CustomerID) as TotalCustomers,
    ROUND(AVG(TotalRevenue), 2) as AvgRevenue
FROM (
    SELECT CustomerID,
           COUNT(DISTINCT InvoiceNo) as TotalOrders,
           SUM(TotalPrice) as TotalRevenue
    FROM orders
    GROUP BY CustomerID
) 
GROUP BY CustomerSegment
ORDER BY TotalCustomers DESC
"""
result5 = pd.read_sql(query5, conn)
print("Customer Segments:")
print(result5)

Customer Segments:
  CustomerSegment  TotalCustomers  AvgRevenue
0      2-5 orders            1964     1083.33
1         1 order            1505      365.44
2     6-10 orders             533     2789.26
3      10+ orders             332    12937.16


## Query 6 — Revenue by Day of Week
Which days generate the most revenue and highest average order value?

In [8]:
# Query 6 - Revenue by day of week
query6 = """
SELECT DayOfWeek,
       ROUND(SUM(TotalPrice), 2) as TotalRevenue,
       COUNT(DISTINCT InvoiceNo) as TotalOrders,
       ROUND(AVG(TotalPrice), 2) as AvgOrderValue
FROM orders
GROUP BY DayOfWeek
ORDER BY TotalRevenue DESC
"""
result6 = pd.read_sql(query6, conn)
print("Revenue by Day of Week:")
print(result6)

Revenue by Day of Week:
   DayOfWeek  TotalRevenue  TotalOrders  AvgOrderValue
0   Thursday    1941119.91         4005          24.59
1    Tuesday    1596464.52         3159          24.37
2  Wednesday    1560549.25         3438          23.02
3     Monday    1327760.48         2835          20.75
4     Friday    1253322.48         2808          23.23
5     Sunday     780238.80         2168          12.77


## Query 7 — Average Order Value by Country
Which countries have the highest spend per order (minimum 20 orders)?

In [9]:
# Query 7 - Average order value by country
query7 = """
SELECT Country,
       ROUND(AVG(TotalPrice), 2) as AvgOrderValue,
       ROUND(SUM(TotalPrice), 2) as TotalRevenue,
       COUNT(DISTINCT InvoiceNo) as TotalOrders
FROM orders
GROUP BY Country
HAVING TotalOrders > 20
ORDER BY AvgOrderValue DESC
LIMIT 10
"""
result7 = pd.read_sql(query7, conn)
print("Average Order Value by Country:")
print(result7)

Average Order Value by Country:
           Country  AvgOrderValue  TotalRevenue  TotalOrders
0      Netherlands         122.26     283889.34           93
1        Australia         117.04     138103.81           56
2           Sweden          86.25      36828.83           34
3             EIRE          36.25     261888.12          258
4           Norway          30.97      32454.64           32
5      Switzerland          28.97      52441.95           47
6          Finland          28.35      18344.88           40
7  Channel Islands          27.01      20147.54           25
8          Germany          23.76     205381.15          443
9            Spain          23.05      55706.56           88


## Query 8 — Repeat vs One Time Customers
What percentage of customers returned for a second purchase?

In [10]:
# Query 8 - Repeat vs one time customers
query8 = """
SELECT 
    SUM(CASE WHEN TotalOrders = 1 THEN 1 ELSE 0 END) as OneTimeCustomers,
    SUM(CASE WHEN TotalOrders > 1 THEN 1 ELSE 0 END) as RepeatCustomers,
    ROUND(SUM(CASE WHEN TotalOrders = 1 THEN 1 ELSE 0 END) * 100.0 / COUNT(*), 1) as OneTimePct,
    ROUND(SUM(CASE WHEN TotalOrders > 1 THEN 1 ELSE 0 END) * 100.0 / COUNT(*), 1) as RepeatPct
FROM (
    SELECT CustomerID,
           COUNT(DISTINCT InvoiceNo) as TotalOrders
    FROM orders
    GROUP BY CustomerID
)
"""
result8 = pd.read_sql(query8, conn)
print("Repeat vs One Time Customers:")
print(result8)

Repeat vs One Time Customers:
   OneTimeCustomers  RepeatCustomers  OneTimePct  RepeatPct
0              1505             2829        34.7       65.3


In [11]:
conn.close()
print("Database connection closed!")

Database connection closed!
